# New Section

In [6]:
import argparse
import gzip
import json
import os
import time

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, precision_recall_fscore_support)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC

In [7]:
CLASS_NAMES = {
    "digits": [str(i) for i in range(10)],
    "fashion": ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
                "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"],
}


In [8]:
# ----------------------------------------------------------------------------
# Data loading
# ----------------------------------------------------------------------------
def _open(path):
    if os.path.exists(path + ".gz"):
        return gzip.open(path + ".gz", "rb")
    return open(path, "rb")


def _read_idx_images(path):
    with _open(path) as f:
        data = f.read()
    n = int.from_bytes(data[4:8], "big")
    rows = int.from_bytes(data[8:12], "big")
    cols = int.from_bytes(data[12:16], "big")
    return np.frombuffer(data, dtype=np.uint8, offset=16).reshape(n, rows * cols)


def _read_idx_labels(path):
    with _open(path) as f:
        data = f.read()
    return np.frombuffer(data, dtype=np.uint8, offset=8).astype(np.int64)

In [9]:
def load_dataset(data_dir):
    xtrain = _read_idx_images(os.path.join(data_dir, "train-images-idx3-ubyte"))
    ytrain = _read_idx_labels(os.path.join(data_dir, "train-labels-idx1-ubyte"))
    xtest = _read_idx_images(os.path.join(data_dir, "t10k-images-idx3-ubyte"))
    ytest = _read_idx_labels(os.path.join(data_dir, "t10k-labels-idx1-ubyte"))
    return xtrain, xtest, ytrain, ytest


def standardize(xtrain, xtest):
    """Map pixel intensities from [0, 255] to [0, 1]."""
    return xtrain.astype(np.float64) / 255.0, xtest.astype(np.float64) / 255.0



In [10]:
# Models
# ----------------------------------------------------------------------------
def build_models():
    """The five classifiers. Hyperparameters are standard, untuned choices so
    the comparison reflects the model families rather than tuning effort."""
    return {
        "KNN": KNeighborsClassifier(
            n_neighbors=5, n_jobs=-1),                 # KNN has no randomness
        "Logistic Regression": LogisticRegression(
            max_iter=1000, random_state=RANDOM_STATE),
        "Linear SVM": SVC(
            kernel="linear", C=1.0, random_state=RANDOM_STATE),
        "Kernel SVM (RBF)": SVC(
            kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE),
        "Neural Network (MLP)": MLPClassifier(
            hidden_layer_sizes=(256, 128), activation="relu", solver="adam",
            alpha=1e-4, batch_size=128, learning_rate_init=1e-3,
            max_iter=100, early_stopping=True, validation_fraction=0.1,
            n_iter_no_change=10, random_state=RANDOM_STATE),
    }


In [3]:
"""
Comparing KNN, Logistic Regression, Linear SVM, Kernel (RBF) SVM and a
Neural Network (MLP) on MNIST Digits and Fashion-MNIST.

  * Random seed: 6740 (passed as random_state everywhere it is accepted)
  * Standardisation: pixel values mapped from [0, 255] to [0, 1]
  * Data shapes: xtrain (60000, 784), xtest (10000, 784),
                 ytrain (60000,),     ytest (10000,)
  * Metrics: per-class precision, recall, F1 (+ accuracy, macro & weighted avg)

Usage (CSV files)
-----------------
    python mnist_classifier_comparison.py --dataset fashion \
        --train-csv fashion-mnist_train.csv --test-csv fashion-mnist_test.csv

    # or point at a folder; any *train*.csv / *test*.csv in it is picked up
    python mnist_classifier_comparison.py --dataset fashion --data-dir .

Each CSV row is: label, pixel1, ..., pixel784 (values 0-255). A header row
such as 'label,pixel1,...' is detected and skipped automatically.

Usage (IDX files)
-----------------
    python mnist_classifier_comparison.py --dataset digits --data-dir data/mnist
A folder with no train/test CSVs is read as the four standard IDX files:
    train-images-idx3-ubyte(.gz)  train-labels-idx1-ubyte(.gz)
    t10k-images-idx3-ubyte(.gz)   t10k-labels-idx1-ubyte(.gz)

If you already have xtrain/xtest/ytrain/ytest in memory (e.g. from a course
loader), skip load_dataset() and call run_all(xtrain, xtest, ytrain, ytest, ...).
"""

import argparse
import gzip
import json
import os
import time

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, precision_recall_fscore_support)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC

RANDOM_STATE = 6740

CLASS_NAMES = {
    "digits": [str(i) for i in range(10)],
    "fashion": ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
                "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"],
}


# ----------------------------------------------------------------------------
# Data loading
# ----------------------------------------------------------------------------
def _open(path):
    if os.path.exists(path + ".gz"):
        return gzip.open(path + ".gz", "rb")
    return open(path, "rb")


def _read_idx_images(path):
    with _open(path) as f:
        data = f.read()
    n = int.from_bytes(data[4:8], "big")
    rows = int.from_bytes(data[8:12], "big")
    cols = int.from_bytes(data[12:16], "big")
    return np.frombuffer(data, dtype=np.uint8, offset=16).reshape(n, rows * cols)


def _read_idx_labels(path):
    with _open(path) as f:
        data = f.read()
    return np.frombuffer(data, dtype=np.uint8, offset=8).astype(np.int64)


def load_dataset(data_dir):
    xtrain = _read_idx_images(os.path.join(data_dir, "train-images-idx3-ubyte"))
    ytrain = _read_idx_labels(os.path.join(data_dir, "train-labels-idx1-ubyte"))
    xtest = _read_idx_images(os.path.join(data_dir, "t10k-images-idx3-ubyte"))
    ytest = _read_idx_labels(os.path.join(data_dir, "t10k-labels-idx1-ubyte"))
    return xtrain, xtest, ytrain, ytest


def load_csv(path):
    """Read one MNIST-style CSV: label in the first column, then 784 pixel
    columns (0-255). Works with or without a header row, e.g. the Kaggle
    files whose header is 'label,pixel1,...,pixel784'."""
    with open(path) as f:
        first = f.readline().split(",")
    try:
        float(first[0])
        skip = 0                                   # first line is data
    except ValueError:
        skip = 1                                   # first line is a header
    try:
        import pandas as pd                        # fast path
        data = pd.read_csv(path, header=None, skiprows=skip,
                           dtype=np.int64).to_numpy()
    except ImportError:
        data = np.loadtxt(path, delimiter=",", skiprows=skip, dtype=np.int64)
    y = data[:, 0].astype(np.int64)
    x = data[:, 1:].astype(np.uint8)
    if x.shape[1] != 784:
        raise ValueError(f"{path}: expected 784 pixel columns, found {x.shape[1]}")
    return x, y


def load_csv_dataset(train_csv, test_csv):
    xtrain, ytrain = load_csv(train_csv)
    xtest, ytest = load_csv(test_csv)
    return xtrain, xtest, ytrain, ytest


def find_csvs(data_dir):
    """Find the train/test CSVs in a folder, accepting either '-' or '_'
    in the name (fashion-mnist_train.csv, fashion_mnist_test.csv,
    mnist_train.csv, ...)."""
    found = {}
    for name in sorted(os.listdir(data_dir)):
        low = name.lower()
        if not low.endswith(".csv"):
            continue
        for split in ("train", "test"):
            if split in low and split not in found:
                found[split] = os.path.join(data_dir, name)
    if len(found) == 2:
        return found["train"], found["test"]
    return None


def standardize(xtrain, xtest):
    """Map pixel intensities from [0, 255] to [0, 1]."""
    return xtrain.astype(np.float64) / 255.0, xtest.astype(np.float64) / 255.0


# ----------------------------------------------------------------------------
# Models
# ----------------------------------------------------------------------------
def build_models():
    """The five classifiers. Hyperparameters are standard, untuned choices so
    the comparison reflects the model families rather than tuning effort."""
    return {
        "KNN": KNeighborsClassifier(
            n_neighbors=5, n_jobs=-1),                 # KNN has no randomness
        "Logistic Regression": LogisticRegression(
            max_iter=1000, random_state=RANDOM_STATE),
        "Linear SVM": SVC(
            kernel="linear", C=1.0, random_state=RANDOM_STATE),
        "Kernel SVM (RBF)": SVC(
            kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE),
        "Neural Network (MLP)": MLPClassifier(
            hidden_layer_sizes=(256, 128), activation="relu", solver="adam",
            alpha=1e-4, batch_size=128, learning_rate_init=1e-3,
            max_iter=100, early_stopping=True, validation_fraction=0.1,
            n_iter_no_change=10, random_state=RANDOM_STATE),
    }


# ----------------------------------------------------------------------------
# Evaluation
# ----------------------------------------------------------------------------
def evaluate(name, model, xtrain, xtest, ytrain, ytest, class_names):
    t0 = time.time()
    model.fit(xtrain, ytrain)
    train_time = time.time() - t0

    t0 = time.time()
    ypred = model.predict(xtest)
    pred_time = time.time() - t0

    p, r, f1, support = precision_recall_fscore_support(ytest, ypred, zero_division=0)
    report_text = classification_report(ytest, ypred, target_names=class_names,
                                        digits=4, zero_division=0)
    report_dict = classification_report(ytest, ypred, target_names=class_names,
                                        digits=4, zero_division=0, output_dict=True)
    return {
        "model": name,
        "accuracy": float(accuracy_score(ytest, ypred)),
        "per_class": [
            {"class": class_names[i], "precision": float(p[i]), "recall": float(r[i]),
             "f1": float(f1[i]), "support": int(support[i])}
            for i in range(len(class_names))
        ],
        "macro_avg": report_dict["macro avg"],
        "weighted_avg": report_dict["weighted avg"],
        "confusion_matrix": confusion_matrix(ytest, ypred).tolist(),
        "train_time_s": train_time,
        "predict_time_s": pred_time,
        "report_text": report_text,
    }


def run_all(xtrain, xtest, ytrain, ytest, dataset, out_dir="results", only=None):
    os.makedirs(out_dir, exist_ok=True)
    class_names = CLASS_NAMES[dataset]
    xtrain, xtest = standardize(xtrain, xtest)
    print(f"\n=== {dataset.upper()} ===")
    print(f"xtrain {xtrain.shape}, xtest {xtest.shape}, "
          f"ytrain {ytrain.shape}, ytest {ytest.shape}, "
          f"pixel range [{xtrain.min():.1f}, {xtrain.max():.1f}]")

    results = []
    for name, model in build_models().items():
        if only and name not in only:
            continue
        fname = os.path.join(out_dir, f"{dataset}_{name.split(' ')[0].lower()}"
                             f"{'_rbf' if 'RBF' in name else ''}.json")
        if os.path.exists(fname):                       # resume support
            with open(fname) as f:
                res = json.load(f)
            print(f"[cached] {name}: accuracy {res['accuracy']:.4f}")
        else:
            print(f"\nTraining {name} ...", flush=True)
            res = evaluate(name, model, xtrain, xtest, ytrain, ytest, class_names)
            with open(fname, "w") as f:
                json.dump(res, f, indent=2)
        print(f"\n{name} | {dataset} | accuracy {res['accuracy']:.4f} | "
              f"train {res['train_time_s']:.1f}s | predict {res['predict_time_s']:.1f}s")
        print(res["report_text"], flush=True)
        results.append(res)

    print(f"\nSummary ({dataset})")
    print(f"{'Model':<24}{'Accuracy':>10}{'Macro P':>10}{'Macro R':>10}{'Macro F1':>10}")
    for res in sorted(results, key=lambda r: -r["accuracy"]):
        m = res["macro_avg"]
        print(f"{res['model']:<24}{res['accuracy']:>10.4f}{m['precision']:>10.4f}"
              f"{m['recall']:>10.4f}{m['f1-score']:>10.4f}")
    return results


if __name__ == "__main__":
    # Notebook Mode config parameters
    DATASET = "fashion"
    DATA_DIR = "/content"
    TRAIN_CSV = "/content/fashion-mnist_train.csv"
    TEST_CSV = "/content/fashion-mnist_test.csv"
    OUT_DIR = "results"
    ONLY = ["KNN", "Logistic Regression"] # Let's run a subset first for speed

    np.random.seed(RANDOM_STATE)

    if TRAIN_CSV and TEST_CSV and os.path.exists(TRAIN_CSV) and os.path.exists(TEST_CSV):
        csvs = (TRAIN_CSV, TEST_CSV)
    else:
        csvs = find_csvs(DATA_DIR)

    if csvs:
        print(f"Reading {csvs[0]} and {csvs[1]}")
        xtrain, xtest, ytrain, ytest = load_csv_dataset(*csvs)
    else:
        xtrain, xtest, ytrain, ytest = load_dataset(DATA_DIR)

    run_all(xtrain, xtest, ytrain, ytest, DATASET, OUT_DIR, ONLY)

Reading /content/fashion-mnist_train.csv and /content/fashion-mnist_test.csv

=== FASHION ===
xtrain (60000, 784), xtest (10000, 784), ytrain (60000,), ytest (10000,), pixel range [0.0, 1.0]

Training KNN ...

KNN | fashion | accuracy 0.8589 | train 0.2s | predict 41.1s
              precision    recall  f1-score   support

 T-shirt/top     0.7660    0.8740    0.8164      1000
     Trouser     0.9857    0.9650    0.9752      1000
    Pullover     0.7530    0.8080    0.7795      1000
       Dress     0.9130    0.8820    0.8973      1000
        Coat     0.7872    0.7990    0.7931      1000
      Sandal     0.9951    0.8160    0.8967      1000
       Shirt     0.6819    0.5830    0.6286      1000
     Sneaker     0.8718    0.9450    0.9069      1000
         Bag     0.9774    0.9530    0.9651      1000
  Ankle boot     0.8828    0.9640    0.9216      1000

    accuracy                         0.8589     10000
   macro avg     0.8614    0.8589    0.8580     10000
weighted avg     0.8614  

In [4]:
def build_models():
    mlp = MLPClassifier(
        hidden_layer_sizes=(256, 128),  # 784 -> 256 -> 128 -> 10 output units
        activation="relu",              # max(0, z): fast, avoids vanishing gradients
        solver="adam",                  # adaptive SGD; works well on 60k images
        alpha=1e-4,                     # L2 penalty on the weights (regularisation)
        batch_size=128,                 # images per gradient step
        learning_rate_init=1e-3,        # Adam step size
        max_iter=100,                   # upper limit on epochs
        early_stopping=True,            # hold out 10% of train, stop when
        validation_fraction=0.1,        #   validation accuracy stops improving
        n_iter_no_change=10,            #   for 10 epochs in a row
        random_state=RANDOM_STATE,      # weight init, shuffling, validation split
    )

    linear_svm = SVC(
        kernel="linear",                # decision function w.x + b (flat boundary)
        C=1.0,                          # margin vs. training error trade-off:
                                        #   small C = wider margin, more errors allowed
        decision_function_shape="ovr",  # sklearn trains one-vs-one (45 pairs for
                                        #   10 classes) and reports ovr-style scores
        random_state=RANDOM_STATE,      # only used if probability=True; set for completeness
    )

    rbf_svm = SVC(
        kernel="rbf",                   # K(x, x') = exp(-gamma * ||x - x'||^2)
        C=1.0,
        gamma="scale",                  # gamma = 1 / (n_features * X.var()):
                                        #   larger gamma = more local, wigglier boundary
        decision_function_shape="ovr",
        random_state=RANDOM_STATE,
    )
    return {"Neural Network (MLP)": mlp,
            "Linear SVM": linear_svm,
            "Kernel SVM (RBF)": rbf_svm}


In [8]:
# 4. Train, predict, report
# ----------------------------------------------------------------------------
def train_and_report(name, model, xtrain, ytrain, xtest, ytest, class_names):
    t0 = time.time()
    model.fit(xtrain, ytrain)
    train_s = time.time() - t0

    t0 = time.time()
    ypred = model.predict(xtest)
    pred_s = time.time() - t0

    acc = accuracy_score(ytest, ypred)
    print("=" * 66)
    print(f"{name}")
    print(f"  trained on {len(ytrain):,} images in {train_s:.1f}s, "
          f"predicted {len(ytest):,} in {pred_s:.1f}s")
    if isinstance(model, MLPClassifier):
        print(f"  epochs run: {model.n_iter_}, "
              f"best validation accuracy: {max(model.validation_scores_):.4f}")
    else:
        print(f"  support vectors: {model.n_support_.sum():,} "
              f"({model.n_support_.sum() / len(ytrain):.1%} of the training set)")
    print(f"  test accuracy: {acc:.4f}\n")
    print(classification_report(ytest, ypred, target_names=class_names,
                                digits=4, zero_division=0))
    print("Confusion matrix (rows = true class, columns = predicted):")
    print(confusion_matrix(ytest, ypred))
    print()
    return {"model": name, "accuracy": acc, "train_s": train_s, "predict_s": pred_s}


def main():
    import sys
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", choices=["digits", "fashion"], default="fashion")
    ap.add_argument("--train-csv", default="/content/fashion-mnist_train.csv")
    ap.add_argument("--test-csv", default="/content/fashion-mnist_test.csv")
    ap.add_argument("--svm-train-size", type=int, default=2000,
                    help="train the SVMs on a random subset of this size "
                         "(default: all 60,000 images)")

    # Notebook mode bypass: check if we are in interactive Google Colab or IPython session
    if any("kernel_launcher" in arg for arg in sys.argv) or any(arg == "-f" for arg in sys.argv):
        # Running in Colab/Jupyter; parse empty list to avoid parsing Colab's sys.argv flags
        args = ap.parse_args([])
        print("[Notebook Mode Enabled] Using default paths.")
    else:
        # Running via terminal command-line
        args = ap.parse_args()

    xtrain, ytrain = load_csv(args.train_csv)
    xtest, ytest = load_csv(args.test_csv)
    print(f"xtrain {xtrain.shape}, xtest {xtest.shape}, "
          f"ytrain {ytrain.shape}, ytest {ytest.shape}")

    xtrain, xtest = standardize(xtrain, xtest)
    print(f"pixel range after scaling: [{xtrain.min():.1f}, {xtrain.max():.1f}]\n")

    # Optional SVM subset: SVC training time grows roughly quadratically with
    # the number of training images, so a subset is handy for quick checks.
    if args.svm_train_size:
        rng = np.random.RandomState(RANDOM_STATE)
        idx = rng.choice(len(ytrain), args.svm_train_size, replace=False)
        x_svm, y_svm = xtrain[idx], ytrain[idx]
    else:
        x_svm, y_svm = xtrain, ytrain

    names = CLASS_NAMES[args.dataset]
    results = []
    for name, model in build_models().items():
        xt, yt = (xtrain, ytrain) if isinstance(model, MLPClassifier) else (x_svm, y_svm)
        results.append(train_and_report(name, model, xt, yt, xtest, ytest, names))

    print("=" * 66)
    print(f"Summary ({args.dataset})")
    print(f"{'Model':<24}{'Accuracy':>10}{'Train (s)':>12}{'Predict (s)':>13}")
    for r in sorted(results, key=lambda r: -r["accuracy"]):
        print(f"{r['model']:<24}{r['accuracy']:>10.4f}{r['train_s']:>12.1f}{r['predict_s']:>13.1f}")


if __name__ == "__main__":
    main()

[Notebook Mode Enabled] Using default paths.
xtrain (60000, 784), xtest (10000, 784), ytrain (60000,), ytest (10000,)
pixel range after scaling: [0.0, 1.0]

Neural Network (MLP)
  trained on 60,000 images in 279.4s, predicted 10,000 in 0.2s
  epochs run: 40, best validation accuracy: 0.9018
  test accuracy: 0.9007

              precision    recall  f1-score   support

 T-shirt/top     0.8674    0.8310    0.8488      1000
     Trouser     0.9859    0.9810    0.9835      1000
    Pullover     0.8761    0.8060    0.8396      1000
       Dress     0.8975    0.9190    0.9081      1000
        Coat     0.8124    0.8790    0.8444      1000
      Sandal     0.9757    0.9630    0.9693      1000
       Shirt     0.7321    0.7350    0.7335      1000
     Sneaker     0.9442    0.9480    0.9461      1000
         Bag     0.9704    0.9830    0.9767      1000
  Ankle boot     0.9497    0.9620    0.9558      1000

    accuracy                         0.9007     10000
   macro avg     0.9011    0.9007